# C-DOT 5G Digital Twin Workshop

**90 minutes · individual JupyterHub lab**

All topology, traffic, telemetry, forecasts and outcomes are synthetic.

In [ ]:
from pathlib import Path
import json, os, sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import HTML, IFrame, Javascript, display
from workshop import runtime
from workshop.solver import teaching_problem, solve_teaching_lp, compare_lp_solvers
from workshop.replay import export_replay
from workshop.report import export_report
display(HTML('<style>.lab{border-left:6px solid #18a3a7;padding:16px;background:#edf8f8}.hint{border:1px solid #d8b55b;padding:10px}.jp-Notebook{max-width:1120px;margin:auto}</style><div class="lab"><b>SYNTHETIC · INDIVIDUAL WORKSPACE</b><h2>Optimize → parallel solve → simulate → analyze → experience</h2>No dashboard credentials. No policy-publication access. No established-session migration.</div>'))


## 01 · PREFLIGHT

Verify identity, PBS, private storage, solver commands/imports and WebGL2. A missing ParaSCIP capability is a blocking readiness failure, even though archived evidence remains viewable.

In [ ]:
checks = runtime.preflight(); display(checks)
display(Javascript("document.body.dataset.webgl2 = !!document.createElement('canvas').getContext('webgl2'); console.log('WebGL2', document.body.dataset.webgl2)"))

<div class='hint'><b>Two-minute hint</b> · `solver_readiness` must be `ready` for the live solver track. The browser check is independent of Python.</div>

In [ ]:
checks = runtime.preflight(); display(checks)

## 02 · OPTIMIZE

Formulate continuous allocation, capacity, eligibility and overload-slack variables. Solve exactly the same LP with HiGHS and SCIP; weights are normalized allocations.

In [ ]:
problem = teaching_problem(demand_mbps=260)
highs = solve_teaching_lp(problem, solver='highs'); display(highs.to_dict())
try:
    comparison = compare_lp_solvers(problem); display(comparison)
except RuntimeError as error:
    print(error); print('LIVE SCIP TRACK BLOCKED — no solver substitution')

<div class='hint'><b>Two-minute hint</b> · Increase `demand_mbps` or reduce one residual capacity. HiGHS is for the tiny LP; SCIP equivalence is checked within tolerance.</div>

In [ ]:
problem = teaching_problem(demand_mbps=260)
highs = solve_teaching_lp(problem, solver='highs'); display(highs.to_dict())
try:
    comparison = compare_lp_solvers(problem); display(comparison)
except RuntimeError as error:
    print(error); print('LIVE SCIP TRACK BLOCKED — no solver substitution')

## 03 · PARALLEL SOLVER

Load the frozen 24-UPF/96-group binary assignment/activation MIP. Submit one participant SCIP job. Observe—not submit—the presenter’s reserved two-node ParaSCIP job.

In [ ]:
mip = json.loads((ROOT/'workshop/data/national_assignment_mip.json').read_text()); print(len(mip['upfs']), len(mip['groups']))
scip_job = runtime.submit_pbs(ROOT/'pbs/workshop_solver.pbs', variables={'WORKSHOP_OUTPUT_ROOT':checks['personal_root'],'WORKSHOP_SEED':'20260822'}); display(scip_job)
print('Presenter ParaSCIP status:', os.environ.get('CDOT_PARASCIP_STATUS_JSON','archived fallback — presenter controls submission'))

<div class='hint'><b>Two-minute hint</b> · ParaSCIP is inappropriate for the three-variable LP. Compare incumbent, dual bound, gap and fixed seed on the larger frozen MIP.</div>

In [ ]:
scip_job = runtime.submit_pbs(ROOT/'pbs/workshop_solver.pbs', variables={'WORKSHOP_OUTPUT_ROOT':str(checks['personal_root']),'WORKSHOP_SEED':'20260822'}); display(scip_job)

## 04 · SIMULATE

Choose one bounded controller and deterministic seed, then submit a private five-simulated-minute shard as one one-node PBS job. The cluster provides scenario breadth; a simulation is not spread across 160 nodes.

In [ ]:
controller='static'; seed=20260822
sim_job=runtime.submit_pbs(ROOT/'pbs/workshop_simulator.pbs',variables={'WORKSHOP_OUTPUT_ROOT':checks['personal_root'],'WORKSHOP_SEED':str(seed),'WORKSHOP_CONTROLLER':controller}); display(sim_job)

<div class='hint'><b>Two-minute hint</b> · Use one of `static`, `reactive`, or `predictive`. Keep the seed integer and your output root private.</div>

In [ ]:
controller='static'; seed=20260822
sim_job=runtime.submit_pbs(ROOT/'pbs/workshop_simulator.pbs',variables={'WORKSHOP_OUTPUT_ROOT':checks['personal_root'],'WORKSHOP_SEED':str(seed),'WORKSHOP_CONTROLLER':controller}); display(sim_job)

## 05 · ANALYZE

Load the actual Parquet contract and inspect offered/carried traffic, overload, loss, utilization, policy IDs, solver status and decision latency. Missing decision-trace fields remain explicitly unavailable.

In [ ]:
run_path=runtime.supplied_result('workshop-run.parquet')
analysis=runtime.analyze_parquet(run_path); display(HTML(runtime.metric_svg(analysis))); display({k:v for k,v in analysis.items() if k!='series'})

<div class='hint'><b>Two-minute hint</b> · Offered demand is independent of carriage. Do not train demand forecasts on constrained carried traffic.</div>

In [ ]:
run_path=runtime.supplied_result('workshop-run.parquet'); analysis=runtime.analyze_parquet(run_path); display(HTML(runtime.metric_svg(analysis))); display({k:v for k,v in analysis.items() if k!='series'})

## 06 · EXPERIENCE + REPORT

Export `twin-replay/1.0`, open the full Three.js replay, run the unsafe-policy drill, and produce `WorkshopReport.json` plus compact HTML.

In [ ]:
replay_path=Path(checks['personal_root'])/'twin-replay.json'; replay=export_replay(run_path,ROOT/'configs/workshop_short_scenario.json',replay_path,max_frames=120)
assert all(frame['causality']['existing_sessions_anchored'] for frame in replay['frames'])
advisory='We would deploy this in advisory mode only after cluster solver, SMF/EMS hook, security, and matched-evidence gates pass.'
report=export_report(Path(checks['personal_root'])/'reports',participant_id=checks['user'],solver=highs.to_dict(),simulation={k:v for k,v in analysis.items() if k!='series'},replay_path=str(replay_path),advisory_pilot_sentence=advisory); display(report)

<div class='hint'><b>Two-minute hint</b> · The replay URL is `/twin?replay=...`. Policy changes affect future-session particles; established particles remain anchored.</div>

In [ ]:
replay_path=Path(checks['personal_root'])/'twin-replay.json'; export_replay(run_path,ROOT/'configs/workshop_short_scenario.json',replay_path,max_frames=120)
advisory='We would deploy this in advisory mode only after cluster solver, SMF/EMS hook, security, and matched-evidence gates pass.'
report=export_report(Path(checks['personal_root'])/'reports',participant_id=checks['user'],solver=highs.to_dict(),simulation=analysis,replay_path=str(replay_path),advisory_pilot_sentence=advisory); display(report)
prefix=os.environ.get('JUPYTERHUB_SERVICE_PREFIX','/')
replay_url=f'{prefix}files/{replay_path.relative_to(ROOT).as_posix()}'
twin_url=os.environ.get('CDOT_TWIN_URL',f'{prefix}proxy/8010/twin')
display(IFrame(src=f'{twin_url}?replay={replay_url}',width='100%',height=680))

## Evidence boundary

The guided 30-pair story reports **+10.52%** in its matched scope. Later national-scale control-science evidence did **not** promote MPC; **Static remains production-safe**. Neither result is live C-DOT evidence.

Complete: **We would deploy this in advisory mode only after ___**.